# Cross-cohort Reactome pathway recurrence — GradientExplainer feature importance

Publication-quality **cross-cohort Reactome pathway recurrence dot plots** summarizing
GradientExplainer feature importance across six TCGA cohorts (UCEC, LIHC, KIRC, THCA,
LUAD, BRCA), computed at two levels of the biological hierarchy:

- **Direct-L1 variant**: each cohort's top-20 L1 Reactome pathways, selected directly
  from the L1 SHAP attributions (sqrt(n_L2_children) normalization).
- **Direct-L2 variant**: each cohort's top-20 L2 Reactome pathways, selected directly
  from the L2 SHAP attributions (sqrt(gene-count) normalization, consistent with the
  per-cohort Sankey figures).

**Interpretation note.** Feature importance was calculated independently within each
cohort using GradientExplainer, with the normal samples from that cohort used as the
explainer background. Raw attribution magnitudes are **not** comparable across
cohorts. All ranking and normalization first occurs *within* cohort. The purpose of
this figure is to show whether independently derived feature-importance results
converge on common Reactome biological programs.

**Figures.** For each variant, a main-text figure (pathways recurring in >=2 cohorts,
capped at ~20 rows) and a supplementary figure (all pathways) are generated, both
colored by within-cohort rank (rank 1 = darkest).

Outputs: vector PDF/SVG + 600-dpi PNG/TIFF, plus a processed table and a
traceability CSV per variant.


In [ ]:
from __future__ import annotations

from pathlib import Path
from textwrap import fill

import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from matplotlib.cm import ScalarMappable
from matplotlib.colors import Normalize

%matplotlib inline

# Publication style: white background, minimal grid, clean sans-serif typography.
mpl.rcParams.update({
    "font.family": "sans-serif",
    "font.sans-serif": ["Arial", "Helvetica", "DejaVu Sans"],
    "font.size": 9,
    "axes.titlesize": 10,
    "axes.labelsize": 9,
    "xtick.labelsize": 8,
    "ytick.labelsize": 7.5,
    "legend.fontsize": 7.5,
    "figure.dpi": 120,
    "savefig.dpi": 600,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.facecolor": "white",
    "figure.facecolor": "white",
    "savefig.facecolor": "white",
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
})

## Configuration

Edge file, SHAP output directory, per-cohort selection sizes, and output location.

The SHAP directory must contain one sub-directory per cohort (named by the cohort
code), each holding that cohort's GradientExplainer parquet exports:

    {SHAP_DIR}/{cohort}/{cohort}_l1_pathway_importance.parquet
    {SHAP_DIR}/{cohort}/{cohort}_l2_pathway_importance.parquet

Each parquet is long-format: one row per (sample, pathway) pair with an
`importance` column (the per-sample mean-|SHAP| attribution to that pathway).


In [ ]:
# Anchor all paths to the repo root so the notebook runs from any kernel cwd.
_REPO_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").is_file())

EDGES_PATH = _REPO_ROOT / "resources" / "masks" / "edges.tsv"
SHAP_DIR = _REPO_ROOT / "outputs" / "shap" / "per_cohort"

TOP_K_L1 = 20  # per-cohort top-20 L1 pathways (direct-L1 variant)
TOP_K_L2 = 20  # per-cohort top-20 L2 pathways (direct-L2 variant)

# Normalization mode for L2 pathway importance (must match the per-cohort Sankey
# figures so pathway selection is consistent across manuscript figures):
#   "none"       - raw mean |SHAP| (total importance; large pathways favored)
#   "sqrt"       - divide by sqrt(n_genes) (geometric middle ground; recommended)
#   "gene_count" - divide by n_genes (per-gene importance; small pathways favored)
NORMALIZE_BY = "sqrt"

# Six cohorts for the cross-cohort recurrence figure (column order).
COHORTS = ["UCEC", "LIHC", "KIRC", "THCA", "LUAD", "BRCA"]

# Direct-L1 main figure: L1 pathways recurring in >= 2 cohorts, capped at 20 rows.
L1_MAIN_MIN_COHORTS = 2
L1_MAIN_MAX_ROWS = 20
L1_FIXED_DOT_SIZE = 80.0  # fixed dot size for direct-L1 (no n_L2 available)

DPI = 600

OUTPUT_DIR = _REPO_ROOT / "outputs" / "figures"
FIG_STEM = "condensed_pathway_importance"


## Load the BINN layer edges

`resources/masks/edges.tsv` holds the four sparse layer edge lists of the biological
hierarchy (CpG->gene, gene->L2, L2->L1, L1->L0). Only this small tabular file is
read — never the methylation memmaps.


In [ ]:
edges = pd.read_csv(EDGES_PATH, sep="\t", dtype=str)  # all IDs / names as strings

# Split into the four BINN layer tables.
cpg_to_gene = edges[edges["hidden_layer"] == "cpg_to_gene"]
gene_to_l2 = edges[edges["hidden_layer"] == "gene_to_l2"]
l2_to_l1 = edges[edges["hidden_layer"] == "l2_to_l1"]
l1_to_l0 = edges[edges["hidden_layer"] == "l1_to_l0"]

# Integrity checks against the shipped edge file.
assert (len(cpg_to_gene), len(gene_to_l2), len(l2_to_l1), len(l1_to_l0)) == (209_839, 26_761, 502, 133)
assert l2_to_l1["source"].is_unique, "each L2 pathway has exactly one L1 parent in the BINN edge file."
print(f"OK: {len(edges):,} edges across 4 layers "
      f"(cpg_to_gene {len(cpg_to_gene):,} | gene_to_l2 {len(gene_to_l2):,} "
      f"| l2_to_l1 {len(l2_to_l1):,} | l1_to_l0 {len(l1_to_l0):,})")

## Build hierarchy maps (shared across all cohorts)

These maps are derived from `edges.tsv` and are identical for every cohort. They bridge
SHAP output (indexed by display name) to the Reactome ID hierarchy.

In [ ]:
# Reactome ID -> display name (from edges.tsv).
l1_names = l1_to_l0.drop_duplicates("source").set_index("source")["source_name"]
l2_names = gene_to_l2.drop_duplicates("target").set_index("target")["target_name"]
l0_name = l1_to_l0.drop_duplicates("target").set_index("target")["target_name"]

# Hierarchy maps.
l1_name_of = l1_names                      # L1 Reactome ID -> L1 display name
l0_name_of_l1 = l1_to_l0.drop_duplicates("source").set_index("source")["target_name"]
l1_parent_of_l2 = l2_to_l1.drop_duplicates("source").set_index("source")["target"]  # L2 ID -> L1 ID

gene_counts = gene_to_l2.groupby("target")["source"].nunique()  # L2 ID -> unique gene count

print(f"L1 pathways: {len(l1_names)}  |  L2 pathways: {len(l2_names)}  |  L0 categories: {len(l0_name)}")
print(f"Gene counts: L2 median={int(gene_counts.median())}")


def _normalize_l2_imp(l2_imp: pd.Series) -> pd.Series:
    """Apply gene-count normalization to L2 importance (matching the Sankey)."""
    if NORMALIZE_BY == "gene_count":
        return l2_imp / gene_counts.reindex(l2_imp.index).clip(lower=1)
    elif NORMALIZE_BY == "sqrt":
        return l2_imp / np.sqrt(gene_counts.reindex(l2_imp.index).clip(lower=1))
    return l2_imp  # "none"


## Dot-plot rendering

Transposed (landscape) matrix with **rows = cohorts** and **columns = pathways**.
- **Dot size**: `n_L2` (child count) binned into a small number of interpretable
  size levels, or a fixed size for the direct variants (no child count is
  available). Sizes are kept modest so color/presence is the primary visual signal.
- **Dot color**: within-cohort rank (rank 1 = darkest).
- **Empty cohort x pathway combinations** remain blank (no zero-valued dots).
- **L0 bracket band** (top): pathways are grouped by Reactome L0 superclass. Light
  brackets with bold category labels appear above multi-pathway groups; singleton
  groups show the category name as small plain text. Inter-group whitespace provides
  the primary visual grouping cue. The largest groups appear first (left to right).
- **Recurrence counts**: `n_cohorts / len(COHORTS)` shown below each pathway column.
- **Pathway labels** (bottom): wrapped to ~25 chars and rotated for compactness.
  Long pathway names are abbreviated (full Reactome names are in the
  supplement/traceability CSV).
- **Within-group sort**: pathways are sorted by recurrence (`n_cohorts`) descending
  within each L0 group, so each section reads shared -> specific from left to right.
- **Colorbar and size legend**: placed in a dedicated GridSpec row below the labels
  so they never resize the plot axes.


In [ ]:
# ---- Abbreviated display labels for long Reactome pathway names ----
# Full names are preserved in the traceability CSV and processed table.
# Only pathways with names > 35 characters are abbreviated.
LABEL_ABBREVIATIONS = {
    "Metabolism of amino acids and derivatives": "Amino acid metabolism",
    "Metabolism of carbohydrates and carbohydrate derivatives": "Carbohydrate metabolism",
    "Maternal to zygotic transition (MZT)": "Maternal-to-zygotic transition",
    "Transmission across Chemical Synapses": "Chemical synaptic transmission",
    "Metabolism of vitamins and cofactors": "Vitamin & cofactor metabolism",
    "Regulation of beta-cell development": "\u03b2-cell development regulation",
    "Cellular responses to mechanical stimuli": "Mechanical stimulus response",
    "Post-translational protein modification": "Post-translational modification",
    "Protein-protein interactions at synapses": "Synaptic protein interactions",
}


def _wrap_label(text: str, width: int = 45) -> str:
    """Wrap pathway names for axis labels (applies abbreviation first)."""
    text = LABEL_ABBREVIATIONS.get(text, text)
    return fill(text, width=width, break_long_words=True, break_on_hyphens=False)


# ---- Size bins for n_L2 ----
# A small number of easily interpretable levels.
# Reduced sizes so color/presence is the primary signal, L2 count is secondary.
SIZE_BINS = [(1, 1, 30), (2, 3, 60), (4, 6, 100), (7, 99, 150)]
SIZE_LABELS = ["1", "2\u20133", "4\u20136", "\u22657"]


def _n_l2_to_size(n: int) -> float:
    for lo, hi, sz in SIZE_BINS:
        if lo <= n <= hi:
            return float(sz)
    return float(SIZE_BINS[-1][2])


# ---- Inter-group whitespace (in axis units; normal pathway spacing = 1.0) ----
GROUP_GAP = 0.3


def _group_by_l0_sort(
    row_l1_ids: list[str],
    summary_df: pd.DataFrame,
    l0_lookup: pd.Series | None = None,
) -> list[str]:
    """Re-sort pathway IDs grouped by L0 category.

    Groups are ordered by size (largest first), then alphabetically.
    Within each group, pathways are sorted by n_cohorts descending,
    then mean_rank ascending, so each section reads shared \u2192 specific
    from left to right.

    Parameters
    ----------
    l0_lookup : optional Series mapping pathway ID \u2192 L0 category name.
        Defaults to ``l0_name_of_l1`` (L1 \u2192 L0).  Pass an L2 \u2192 L0
        lookup for the direct-L2 variant.
    """
    if l0_lookup is None:
        l0_lookup = l0_name_of_l1
    groups: dict[str, list[str]] = {}
    for lid in row_l1_ids:
        cat = l0_lookup[lid]
        groups.setdefault(cat, []).append(lid)
    ordered_cats = sorted(groups.keys(), key=lambda c: (-len(groups[c]), c))
    _summary = summary_df
    result: list[str] = []
    for cat in ordered_cats:
        group_lids = sorted(
            groups[cat],
            key=lambda lid: (
                -int(_summary.loc[_summary["l1_id"] == lid, "n_cohorts"].iloc[0]),
                float(_summary.loc[_summary["l1_id"] == lid, "mean_rank"].iloc[0]),
            ),
        )
        result.extend(group_lids)
    return result


def draw_dotplot(
    row_l1_ids: list[str],
    matrix_override: pd.DataFrame,
    summary_override: pd.DataFrame,
    suffix: str,
    fig_height: float = 7.2,
    fixed_size: float | None = None,
    name_lookup: pd.Series | None = None,
    l0_lookup: pd.Series | None = None,
) -> None:
    """Draw and save the transposed cross-cohort dot plot (landscape).

    Cohorts are rows (y-axis), pathways are columns (x-axis).
    Pathways are grouped by L0 Reactome category with light bracket labels above.
    Pathway names (rotated 50\u00b0) appear below the recurrence counts.
    Dot size = n_L2 (child count), or a fixed size for the direct variants.
    Dot color = within-cohort rank.
    Recurrence counts (n_cohorts / total) appear below the dot matrix.

    Parameters
    ----------
    row_l1_ids : ordered list of pathway Reactome IDs (sorted by n_cohorts desc, mean_rank asc).
    suffix : filename suffix (e.g. "main_rank").
    fig_height : figure height in inches.
    name_lookup : optional Series mapping pathway ID \u2192 display name.
        Defaults to ``l1_name_of``.  Pass ``l2_names`` for the direct-L2 variant.
    l0_lookup : optional Series mapping pathway ID \u2192 L0 category name.
        Defaults to ``l0_name_of_l1``.  Pass an L2 \u2192 L0 lookup for the
        direct-L2 variant.
    """
    if name_lookup is None:
        name_lookup = l1_name_of
    if l0_lookup is None:
        l0_lookup = l0_name_of_l1
    eff_matrix = matrix_override
    eff_summary = summary_override

    # ---- Re-sort pathways grouped by L0 category ----
    pathway_order = _group_by_l0_sort(row_l1_ids, summary_df=eff_summary, l0_lookup=l0_lookup)
    n_cohorts = len(COHORTS)
    cohort_y = {cohort: i for i, cohort in enumerate(COHORTS)}
    pathway_labels = [_wrap_label(name_lookup[lid], width=25) for lid in pathway_order]

    # ---- Gap-aware x-coordinates with inter-group whitespace ----
    l0_groups: dict[str, list[str]] = {}
    for lid in pathway_order:
        cat = l0_lookup[lid]
        l0_groups.setdefault(cat, []).append(lid)
    ordered_cats = sorted(l0_groups.keys(), key=lambda c: (-len(l0_groups[c]), c))

    pathway_x: dict[str, float] = {}
    group_x_ranges: list[tuple[str, float, float, int]] = []
    x = 0.0
    for gi, cat in enumerate(ordered_cats):
        group_lids = l0_groups[cat]
        x_first = x
        for lid in group_lids:
            pathway_x[lid] = x
            x += 1.0
        x_last = x - 1.0
        group_x_ranges.append((cat, x_first, x_last, len(group_lids)))
        if gi < len(ordered_cats) - 1:
            x += GROUP_GAP
    x_max = x - 1.0
    x_positions = [pathway_x[lid] for lid in pathway_order]

    # ---- Build the plotting matrix (present pairs only) ----
    plot_rows = []
    for l1_id in pathway_order:
        l1_name = name_lookup[l1_id]
        l0_cat = l0_lookup[l1_id]
        for cohort in COHORTS:
            entry = eff_matrix[
                (eff_matrix["l1_id"] == l1_id) & (eff_matrix["cohort"] == cohort)
            ]
            if entry.empty:
                continue
            r = entry.iloc[0]
            plot_rows.append({
                "l1_id": l1_id,
                "L1": l1_name,
                "L0": l0_cat,
                "cohort": cohort,
                "n_L2": int(r["n_L2"]) if "n_L2" in r else 0,
                "L1_rank": int(r["L1_rank"]),
                "n_cohorts": int(r["n_cohorts"]),
            })
    plot_df = pd.DataFrame(plot_rows)

    # ---- Color normalization ----
    color_norm = Normalize(vmin=1, vmax=eff_matrix["L1_rank"].max())
    cmap = plt.get_cmap("mako_r")
    cbar_label = "Within-cohort rank"
    cbar_vals = plot_df["L1_rank"].to_numpy()

    # ---- Figure layout: 5-row, 1-column GridSpec ----
    # Row 0: L0 bracket band (light brackets for multi-pathway groups, plain text for singletons)
    # Row 1: dot matrix (cohort labels on left, no pathway labels on top)
    # Row 2: recurrence counts
    # Row 3: pathway labels (rotated 50\u00b0, on bottom)
    # Row 4: colorbar + size legend
    fig_width = max(14, (x_max + 1) * 0.52)

    fig = plt.figure(figsize=(fig_width, fig_height))
    gs = fig.add_gridspec(
        4, 1,
        height_ratios=[0.07, 1.0, 0.05, 0.04],
        hspace=0.08,
        left=0.06, right=0.98, top=0.93, bottom=0.28,
    )
    ax_l0 = fig.add_subplot(gs[0])
    ax_dot = fig.add_subplot(gs[1])
    ax_recur = fig.add_subplot(gs[2])
    ax_labels = fig.add_subplot(gs[3])
    ax_cbar = fig.add_axes([0.08, 0.06, 0.32, 0.05])
    ax_size = fig.add_axes([0.50, 0.06, 0.40, 0.05])
    ax_cbar.axis("off")
    ax_size.axis("off")

    xlim = (-0.5, x_max + 0.5)

    # ---- L0 bracket band ----
    ax_l0.set_xlim(*xlim)
    ax_l0.set_ylim(0, 1)
    ax_l0.set_xticks([])
    ax_l0.set_yticks([])
    for spine in ax_l0.spines.values():
        spine.set_visible(False)
    bracket_y = 0.35
    tick_y = 0.15
    label_y = 0.60
    for cat, x_start, x_end, n_in_group in group_x_ranges:
        if n_in_group >= 2:
            ax_l0.plot(
                [x_start - 0.4, x_end + 0.4], [bracket_y, bracket_y],
                color="#888888", linewidth=0.7, zorder=2,
            )
            ax_l0.plot(
                [x_start - 0.4, x_start - 0.4], [bracket_y, tick_y],
                color="#888888", linewidth=0.7, zorder=2,
            )
            ax_l0.plot(
                [x_end + 0.4, x_end + 0.4], [bracket_y, tick_y],
                color="#888888", linewidth=0.7, zorder=2,
            )
            center_x = (x_start + x_end) / 2.0
            ax_l0.text(
                center_x, label_y, _wrap_label(cat, width=18),
                ha="center", va="bottom", fontsize=7, fontweight="bold",
                color="#333333",
            )
        else:
            center_x = (x_start + x_end) / 2.0
            ax_l0.text(
                center_x, bracket_y, _wrap_label(cat, width=18),
                ha="center", va="center", fontsize=6,
                color="#666666",
            )

    # ---- Dot matrix ----
    ax_dot.set_xlim(*xlim)
    ax_dot.set_ylim(-0.5, n_cohorts - 0.5)
    ax_dot.invert_yaxis()
    ax_dot.set_xticks(x_positions)
    ax_dot.tick_params(axis="x", length=0, labelbottom=False)
    ax_dot.set_yticks(range(n_cohorts))
    ax_dot.set_yticklabels(COHORTS, fontsize=8, fontweight="bold")
    ax_dot.tick_params(axis="y", length=0)
    ax_dot.yaxis.grid(False)
    ax_dot.xaxis.grid(True, color="#E8E8E8", linewidth=0.5, zorder=0)
    ax_dot.set_axisbelow(True)
    for spine in ax_dot.spines.values():
        spine.set_visible(False)

    # Plot dots
    for _, r in plot_df.iterrows():
        x_val = pathway_x[r["l1_id"]]
        y_val = cohort_y[r["cohort"]]
        size = fixed_size if fixed_size is not None else _n_l2_to_size(r["n_L2"])
        color_val = r["L1_rank"]
        facecolor = cmap(color_norm(color_val))
        ax_dot.scatter(
            x_val, y_val, s=size, c=[facecolor], edgecolors="white", linewidths=0.4,
            zorder=3, marker="o",
        )

    # ---- Recurrence counts (below dot matrix) ----
    ax_recur.set_xlim(*xlim)
    ax_recur.set_ylim(0, 1)
    ax_recur.set_xticks([])
    ax_recur.set_yticks([])
    for spine in ax_recur.spines.values():
        spine.set_visible(False)
    ax_recur.axhline(y=0.9, color="#CCCCCC", linewidth=0.5)
    for lid in pathway_order:
        n_coh = int(eff_summary.loc[eff_summary["l1_id"] == lid, "n_cohorts"].iloc[0])
        ax_recur.text(
            pathway_x[lid], 0.5, f"{n_coh}/{len(COHORTS)}",
            ha="center", va="center", fontsize=7, fontweight="bold",
            color="#333333",
        )

    # ---- Pathway labels (rotated, at bottom) ----
    ax_labels.set_xlim(*xlim)
    ax_labels.set_ylim(0, 1)
    ax_labels.set_xticks(x_positions)
    ax_labels.set_xticklabels(
        pathway_labels, rotation=60, ha="right", va="top",
        rotation_mode="anchor", fontsize=7,
    )
    ax_labels.xaxis.tick_bottom()
    ax_labels.tick_params(axis="x", length=0, pad=1)
    ax_labels.set_yticks([])
    for spine in ax_labels.spines.values():
        spine.set_visible(False)

    # ---- Color bar (dedicated axis — does NOT resize ax_dot) ----
    sm = ScalarMappable(norm=color_norm, cmap=cmap)
    sm.set_array(cbar_vals)
    cax = ax_cbar.inset_axes([0.15, 0.40, 0.70, 0.30])
    cbar = fig.colorbar(sm, cax=cax, orientation="horizontal")
    cbar.set_label(cbar_label, fontsize=7.5)
    cbar.ax.tick_params(labelsize=7)
    cbar.set_ticks([1, eff_matrix["L1_rank"].max()])
    cbar.set_ticklabels(["1 (best)", f"{int(eff_matrix['L1_rank'].max())}"])

    # ---- Size legend (dedicated axis — does NOT resize ax_dot) ----
    # Skipped when fixed_size is set (direct-L1 variant has no n_L2).
    if fixed_size is None:
        size_handles = []
        for (_lo, _hi, sz), lbl in zip(SIZE_BINS, SIZE_LABELS, strict=True):
            size_handles.append(
                ax_size.scatter([], [], s=sz, c="#888888", edgecolors="white",
                                linewidths=0.4, label=lbl, zorder=3)
            )
        ax_size.legend(
            handles=size_handles, title="n_L2 (top-20 L2s→L1)",
            loc="center", ncol=len(SIZE_BINS),
            frameon=False, fontsize=7, title_fontsize=7.5,
            scatterpoints=1, handletextpad=0.3, columnspacing=0.8,
        )

    # ---- Save ----
    stem = f"{FIG_STEM}_{suffix}"
    for ext in ["pdf", "svg", "png", "tiff"]:
        path = OUTPUT_DIR / f"{stem}.{ext}"
        fig.savefig(path, dpi=DPI, bbox_inches="tight", pad_inches=0.15)
    plt.show()
    plt.close(fig)
    print(f"Saved: {stem}.{{pdf,svg,png,tiff}}")

## L1 SHAP variant

A cross-cohort dot plot that reads `*_l1_pathway_importance.parquet` directly,
selecting each cohort's top-20 L1 pathways by **sqrt(n_L2_children)** normalized
importance. This mirrors the L2 pipeline's normalization philosophy (normalize by
the count of units in the immediately-upstream layer) and measures importance at
the L1 layer itself.

Dot size is fixed (no `n_L2` available); color and presence are the only visual
signals. Two variants are generated — main (pathways recurring in >=2 cohorts,
capped at 20 rows) and supplementary (all pathways), both colored by
within-cohort rank.


In [ ]:
# --- L2-child counts for L1 normalization ---
# Mirrors the L2 approach: L2 importance is normalized by sqrt(n_genes) where genes
# are the layer immediately upstream of L2. For L1, the upstream layer is L2, so we
# normalize by sqrt(n_L2_children).
l2_child_counts = l2_to_l1.groupby("target")["source"].nunique()

print(f"L1 pathways: {len(l2_child_counts)}")
print(f"L2 children per L1 — median={int(l2_child_counts.median())}, "
      f"min={int(l2_child_counts.min())}, max={int(l2_child_counts.max())}")

In [ ]:
# --- Per-cohort direct-L1 SHAP loading + aggregation ---
# Inverted L1 name map: display name → Reactome ID
l1_id_of_name = pd.Series(l1_names.index, index=l1_names.values)


def _normalize_l1_imp(l1_imp: pd.Series) -> pd.Series:
    """Normalize L1 importance by sqrt(n_L2_children) — upstream-layer normalization."""
    return l1_imp / np.sqrt(l2_child_counts.reindex(l1_imp.index).clip(lower=1))


def build_cohort_l1_direct_table(cohort: str) -> pd.DataFrame:
    """Return per-cohort top-K L1 pathways directly from L1 SHAP parquet."""
    cohort_dir = SHAP_DIR / cohort
    if not cohort_dir.is_dir():
        raise FileNotFoundError(f"No SHAP output directory for cohort {cohort}: {cohort_dir}")

    # ---- Load this cohort's L1 SHAP parquet ----
    l1_shap = pd.read_parquet(cohort_dir / f"{cohort}_l1_pathway_importance.parquet")

    # ---- Per-cohort mean |SHAP| per L1 pathway (indexed by display name) ----
    l1_imp_by_name = l1_shap.groupby("pathway")["importance"].apply(lambda x: x.abs().mean())

    # ---- Bridge display names → Reactome IDs ----
    l1_imp = pd.Series(
        l1_imp_by_name.reindex(l1_names.values).fillna(0.0).to_numpy(),
        index=l1_names.index,
    )

    # ---- Raw (un-normalized) importance for the traceability CSV ----
    l1_imp_raw = l1_imp.copy()

    # ---- Normalize by sqrt(n_L2_children) ----
    l1_imp_norm = _normalize_l1_imp(l1_imp)

    # ---- Global top-K L1 by normalized importance ----
    top_l1 = l1_imp_norm.nlargest(TOP_K_L1).index

    # ---- Build the per-cohort L1 table ----
    rows = []
    for l1_id in top_l1:
        rows.append({
            "cohort": cohort,
            "l1_id": l1_id,
            "L1": l1_name_of.get(l1_id, l1_id),
            "L0": l0_name_of_l1.get(l1_id, ""),
            "L1_importance": float(l1_imp_norm[l1_id]),
            "L1_importance_raw": float(l1_imp_raw[l1_id]),
            "L1_rank": 0,  # filled after sorting
        })
    agg = pd.DataFrame(rows)
    agg["L1_rank"] = agg["L1_importance"].rank(method="min", ascending=False).astype(int)

    return agg[["cohort", "l1_id", "L1", "L0", "L1_importance", "L1_importance_raw", "L1_rank"]]


# ---- Build per-cohort tables and assemble the cross-cohort matrix ----
cohort_l1_direct_tables = {c: build_cohort_l1_direct_table(c) for c in COHORTS}
matrix_df_l1_direct = pd.concat(cohort_l1_direct_tables.values(), ignore_index=True)

print(f"Direct-L1 cross-cohort matrix: {len(matrix_df_l1_direct)} cohort × L1 entries "
      f"across {len(COHORTS)} cohorts")
print(f"Unique L1 pathways: {matrix_df_l1_direct['l1_id'].nunique()}")
print(matrix_df_l1_direct.head(8).to_string(index=False))

In [ ]:
# --- Cross-cohort recurrence per L1 (direct-L1) ---

# ---- Cross-cohort recurrence per L1 ----
l1_summary_l1_direct = matrix_df_l1_direct.groupby("l1_id").agg(
    L1=("L1", "first"),
    L0=("L0", "first"),
    n_cohorts=("cohort", "nunique"),
    mean_rank=("L1_rank", "mean"),
).reset_index()

# Merge recurrence info back into the matrix
matrix_df_l1_direct = matrix_df_l1_direct.merge(
    l1_summary_l1_direct[["l1_id", "n_cohorts", "mean_rank"]],
    on="l1_id",
    how="left",
)

# ---- Row ordering: decreasing n_cohorts, then increasing mean_rank ----
l1_summary_l1_direct = l1_summary_l1_direct.sort_values(
    ["n_cohorts", "mean_rank"], ascending=[False, True]
).reset_index(drop=True)

print(f"Direct-L1 pathway recurrence summary (all {len(l1_summary_l1_direct)} L1 pathways):")
print(l1_summary_l1_direct.to_string(index=False))

In [ ]:
# --- L1 traceability table (direct-L1 variant) ---
# One row per selected L1 pathway × cohort, with its L0 mapping and importance values.


def build_cohort_l1_direct_traceability(cohort: str) -> pd.DataFrame:
    """Return per-cohort top-K L1 traceability table."""
    cohort_dir = SHAP_DIR / cohort
    l1_shap = pd.read_parquet(cohort_dir / f"{cohort}_l1_pathway_importance.parquet")
    l1_imp_by_name = l1_shap.groupby("pathway")["importance"].apply(lambda x: x.abs().mean())
    l1_imp = pd.Series(
        l1_imp_by_name.reindex(l1_names.values).fillna(0.0).to_numpy(),
        index=l1_names.index,
    )
    l1_imp_raw = l1_imp.copy()
    l1_imp_norm = _normalize_l1_imp(l1_imp)
    top_l1 = l1_imp_norm.nlargest(TOP_K_L1).index

    l1_rank = pd.Series(
        l1_imp_norm.rank(method="min", ascending=False).astype(int).reindex(top_l1).to_numpy(),
        index=top_l1,
    )

    rows = []
    for l1_id in top_l1:
        rows.append({
            "cohort": cohort,
            "l1_id": l1_id,
            "L1_pathway": l1_name_of.get(l1_id, l1_id),
            "l1_rank_within_cohort": int(l1_rank[l1_id]),
            "l1_importance_norm": float(l1_imp_norm[l1_id]),
            "l1_importance_raw": float(l1_imp_raw[l1_id]),
            "L0": l0_name_of_l1.get(l1_id, ""),
            "n_L2_children": int(l2_child_counts.get(l1_id, 0)),
        })
    return pd.DataFrame(rows)


traceability_l1_direct_df = pd.concat(
    [build_cohort_l1_direct_traceability(c) for c in COHORTS], ignore_index=True
)

traceability_l1_path = OUTPUT_DIR / f"{FIG_STEM}_l1_direct_traceability.csv"
traceability_l1_direct_df.to_csv(traceability_l1_path, index=False)
print(f"L1 direct traceability: {len(traceability_l1_direct_df)} rows "
      f"({TOP_K_L1} × {len(COHORTS)} cohorts)")
print(traceability_l1_direct_df.head(6).to_string(index=False))
print(f"\nSaved: {traceability_l1_path}")

In [ ]:
# --- Row selection: main vs supplementary (direct-L1) ---

main_l1_direct = l1_summary_l1_direct[
    l1_summary_l1_direct["n_cohorts"] >= L1_MAIN_MIN_COHORTS
].copy()
main_l1_direct = main_l1_direct.head(L1_MAIN_MAX_ROWS)
supp_l1_direct = l1_summary_l1_direct.copy()

print(f"Direct-L1 main figure: {len(main_l1_direct)} L1 pathways "
      f"(>= {L1_MAIN_MIN_COHORTS} cohorts, capped at {L1_MAIN_MAX_ROWS})")
print(f"Direct-L1 supplementary figure: {len(supp_l1_direct)} L1 pathways (all)")
print()
print("Direct-L1 main figure rows:")
print(main_l1_direct[["L1", "L0", "n_cohorts", "mean_rank"]].to_string(index=False))

In [ ]:
# --- Generate direct-L1 figure variants ---
# Two variants: main/supplementary, both colored by within-cohort rank.
# Dot size is fixed (no n_L2 available); size legend is hidden.

main_l1_direct_ids = main_l1_direct["l1_id"].tolist()
supp_l1_direct_ids = supp_l1_direct["l1_id"].tolist()

draw_dotplot(
    main_l1_direct_ids, "l1_direct_main_rank",
    matrix_override=matrix_df_l1_direct,
    summary_override=l1_summary_l1_direct,
    fixed_size=L1_FIXED_DOT_SIZE,
)
draw_dotplot(
    supp_l1_direct_ids, "l1_direct_supp_rank",
    matrix_override=matrix_df_l1_direct,
    summary_override=l1_summary_l1_direct,
    fixed_size=L1_FIXED_DOT_SIZE,
)

print(f"\nAll direct-L1 figure variants generated in {OUTPUT_DIR}")


In [ ]:
# --- Processed table + integrity checks (direct-L1) ---

processed_l1_cols = [
    "cohort", "L0", "L1", "L1_importance", "L1_rank",
    "n_cohorts", "mean_rank", "L1_importance_raw",
]
processed_l1_direct_df = matrix_df_l1_direct[processed_l1_cols].sort_values(
    ["n_cohorts", "L1", "cohort"], ascending=[False, True, True]
).reset_index(drop=True)

processed_l1_path = OUTPUT_DIR / f"{FIG_STEM}_l1_direct_processed.csv"
processed_l1_direct_df.to_csv(processed_l1_path, index=False)
print(f"Direct-L1 processed table: {len(processed_l1_direct_df)} rows")
print(processed_l1_direct_df.head(10).to_string(index=False))
print(f"\nSaved: {processed_l1_path}")

# ---- Integrity checks ----
print()

# 1. Max recurrence
max_n_cohorts_l1 = l1_summary_l1_direct["n_cohorts"].max()
assert max_n_cohorts_l1 >= 1, f"Expected at least 1 cohort recurrence, got {max_n_cohorts_l1}"
print(f"OK: max n_cohorts (direct-L1) = {max_n_cohorts_l1}/{len(COHORTS)}")

# 2. L1_importance sum within each cohort equals the sum of that cohort's top-K L1
#    normalized importances.
for cohort in COHORTS:
    cohort_l1_sum = matrix_df_l1_direct[
        matrix_df_l1_direct["cohort"] == cohort
    ]["L1_importance"].sum()
    # Recompute from scratch.
    l1_shap = pd.read_parquet(SHAP_DIR / cohort / f"{cohort}_l1_pathway_importance.parquet")
    l1_imp_by_name = l1_shap.groupby("pathway")["importance"].apply(lambda x: x.abs().mean())
    l1_imp = pd.Series(
        l1_imp_by_name.reindex(l1_names.values).fillna(0.0).to_numpy(),
        index=l1_names.index,
    )
    l1_imp_norm = _normalize_l1_imp(l1_imp)
    topK_sum = l1_imp_norm.nlargest(TOP_K_L1).sum()
    assert np.isclose(cohort_l1_sum, topK_sum), (
        f"{cohort}: L1_importance sum {cohort_l1_sum:.6f} != top-{TOP_K_L1} L1 sum {topK_sum:.6f}"
    )
print(f"OK: L1_importance sums match top-{TOP_K_L1} L1 normalized importance sums for all {len(COHORTS)} cohorts")

# 3. Every L1 in the traceability CSV maps to an L1 in the processed table for that cohort.
merged_l1 = traceability_l1_direct_df.merge(
    matrix_df_l1_direct[["cohort", "l1_id"]].drop_duplicates(),
    on=["cohort", "l1_id"], how="left", indicator=True,
)
orphans_l1 = merged_l1[merged_l1["_merge"] == "left_only"]
assert len(orphans_l1) == 0, (
    f"{len(orphans_l1)} L1s in direct-L1 traceability have no entry in the processed table"
)
print(f"OK: all {len(traceability_l1_direct_df)} direct-L1 traceability rows map to entries in the processed table")

# 4. n_cohorts values are consistent.
check_nc_l1 = matrix_df_l1_direct.groupby("l1_id")["cohort"].nunique().rename("n_cohorts_check")
check_nc_l1 = check_nc_l1.reindex(l1_summary_l1_direct["l1_id"]).to_numpy()
mismatch_l1 = (l1_summary_l1_direct["n_cohorts"].to_numpy() != check_nc_l1)
assert not mismatch_l1.any(), (
    f"n_cohorts mismatch for: {l1_summary_l1_direct.loc[mismatch_l1, 'L1'].tolist()}"
)
print("OK: n_cohorts is consistent between matrix_df_l1_direct and l1_summary_l1_direct")

## L2 SHAP variant

A cross-cohort dot plot that operates directly at the **L2 hidden layer** level.
Each cohort's top-20 L2 pathways are selected by sqrt(gene-count) normalized
GradientExplainer importance (the same normalization as the per-cohort Sankey
figures), measuring cross-cohort recurrence of individual L2 Reactome pathways.

Columns are grouped by **L0 grandparent** (via L2->L1->L0) with bracket labels,
mirroring the L0 grouping used in the L1-level figures. Dot size is fixed (no
child-count analog at the L2 layer); color and presence are the only visual
signals. Two variants are generated — main (pathways recurring in >=2 cohorts,
capped at 20 rows) and supplementary (all pathways), both colored by
within-cohort rank.


In [ ]:
# --- L2 → L0 grandparent lookup (via L2 → L1 → L0) ---
l0_of_l2 = l1_parent_of_l2.map(l0_name_of_l1)

L2_FIXED_DOT_SIZE = 80.0
L2_MAIN_MIN_COHORTS = 2
L2_MAIN_MAX_ROWS = 20


def build_cohort_l2_direct_table(cohort: str) -> pd.DataFrame:
    """Return per-cohort top-K L2 pathways directly from L2 SHAP parquet."""
    cohort_dir = SHAP_DIR / cohort
    if not cohort_dir.is_dir():
        raise FileNotFoundError(f"No SHAP output directory for cohort {cohort}: {cohort_dir}")

    # ---- Load this cohort's L2 SHAP parquet ----
    l2_shap = pd.read_parquet(cohort_dir / f"{cohort}_l2_pathway_importance.parquet")

    # ---- Per-cohort mean |SHAP| per L2 pathway (indexed by display name) ----
    l2_imp_by_name = l2_shap.groupby("pathway")["importance"].apply(lambda x: x.abs().mean())

    # ---- Bridge display names → Reactome IDs ----
    l2_imp = pd.Series(
        l2_imp_by_name.reindex(l2_names.values).fillna(0.0).to_numpy(),
        index=l2_names.index,
    )

    # ---- Raw (un-normalized) importance ----
    l2_imp_raw = l2_imp.copy()

    # ---- Normalize by sqrt(gene_count) ----
    l2_imp_norm = _normalize_l2_imp(l2_imp)

    # ---- Global top-K L2 by normalized importance ----
    top_l2 = l2_imp_norm.nlargest(TOP_K_L2).index

    # ---- Build the per-cohort L2 table ----
    # Column names match draw_dotplot's expectations (l1_id, L1, L0, L1_rank, etc.)
    # so the existing function works without further changes.
    rows = []
    for l2_id in top_l2:
        rows.append({
            "cohort": cohort,
            "l1_id": l2_id,
            "L1": l2_names.get(l2_id, l2_id),
            "L0": l0_of_l2.get(l2_id, ""),
            "L1_importance": float(l2_imp_norm[l2_id]),
            "L1_importance_raw": float(l2_imp_raw[l2_id]),
            "L1_rank": 0,  # filled after sorting
        })
    agg = pd.DataFrame(rows)
    agg["L1_rank"] = agg["L1_importance"].rank(method="min", ascending=False).astype(int)

    return agg[["cohort", "l1_id", "L1", "L0", "L1_importance", "L1_importance_raw", "L1_rank"]]


# ---- Build per-cohort tables and assemble the cross-cohort matrix ----
cohort_l2_direct_tables = {c: build_cohort_l2_direct_table(c) for c in COHORTS}
matrix_df_l2_direct = pd.concat(cohort_l2_direct_tables.values(), ignore_index=True)

print(f"Direct-L2 cross-cohort matrix: {len(matrix_df_l2_direct)} cohort × L2 entries "
      f"across {len(COHORTS)} cohorts")
print(f"Unique L2 pathways: {matrix_df_l2_direct['l1_id'].nunique()}")
print(matrix_df_l2_direct.head(8).to_string(index=False))

In [ ]:
# --- Cross-cohort recurrence per L2 (direct-L2) ---

# ---- Cross-cohort recurrence per L2 ----
l2_summary_l2_direct = matrix_df_l2_direct.groupby("l1_id").agg(
    L1=("L1", "first"),
    L0=("L0", "first"),
    n_cohorts=("cohort", "nunique"),
    mean_rank=("L1_rank", "mean"),
).reset_index()

# Merge recurrence info back into the matrix
matrix_df_l2_direct = matrix_df_l2_direct.merge(
    l2_summary_l2_direct[["l1_id", "n_cohorts", "mean_rank"]],
    on="l1_id",
    how="left",
)

# ---- Row ordering: decreasing n_cohorts, then increasing mean_rank ----
l2_summary_l2_direct = l2_summary_l2_direct.sort_values(
    ["n_cohorts", "mean_rank"], ascending=[False, True]
).reset_index(drop=True)

print(f"Direct-L2 pathway recurrence summary (all {len(l2_summary_l2_direct)} L2 pathways):")
print(l2_summary_l2_direct.head(20).to_string(index=False))

In [ ]:
# --- Row selection: main vs supplementary (direct-L2) ---

main_l2_direct = l2_summary_l2_direct[
    l2_summary_l2_direct["n_cohorts"] >= L2_MAIN_MIN_COHORTS
].copy()
main_l2_direct = main_l2_direct.head(L2_MAIN_MAX_ROWS)
supp_l2_direct = l2_summary_l2_direct.copy()

print(f"Direct-L2 main figure: {len(main_l2_direct)} L2 pathways "
      f"(>= {L2_MAIN_MIN_COHORTS} cohorts, capped at {L2_MAIN_MAX_ROWS})")
print(f"Direct-L2 supplementary figure: {len(supp_l2_direct)} L2 pathways (all)")
print()
print("Direct-L2 main figure rows:")
print(main_l2_direct[["L1", "L0", "n_cohorts", "mean_rank"]].to_string(index=False))

In [ ]:
# --- Generate direct-L2 figure variants ---
# Two variants: main/supplementary, both colored by within-cohort rank.
# Dot size is fixed; size legend is hidden.
# Columns are grouped by L0 grandparent (via l0_of_l2 lookup).

main_l2_direct_ids = main_l2_direct["l1_id"].tolist()
supp_l2_direct_ids = supp_l2_direct["l1_id"].tolist()

draw_dotplot(
    main_l2_direct_ids, "l2_direct_main_rank",
    matrix_override=matrix_df_l2_direct,
    summary_override=l2_summary_l2_direct,
    fixed_size=L2_FIXED_DOT_SIZE,
    name_lookup=l2_names,
    l0_lookup=l0_of_l2,
)
draw_dotplot(
    supp_l2_direct_ids, "l2_direct_supp_rank",
    matrix_override=matrix_df_l2_direct,
    summary_override=l2_summary_l2_direct,
    fixed_size=L2_FIXED_DOT_SIZE,
    name_lookup=l2_names,
    l0_lookup=l0_of_l2,
)

print(f"\nAll direct-L2 figure variants generated in {OUTPUT_DIR}")


In [ ]:
# --- Processed table + integrity checks (direct-L2) ---

processed_l2_cols = [
    "cohort", "L0", "L1", "L1_importance", "L1_rank",
    "n_cohorts", "mean_rank", "L1_importance_raw",
]
processed_l2_direct_df = matrix_df_l2_direct[processed_l2_cols].sort_values(
    ["n_cohorts", "L1", "cohort"], ascending=[False, True, True]
).reset_index(drop=True)

processed_l2_path = OUTPUT_DIR / f"{FIG_STEM}_l2_direct_processed.csv"
processed_l2_direct_df.to_csv(processed_l2_path, index=False)
print(f"Direct-L2 processed table: {len(processed_l2_direct_df)} rows")
print(processed_l2_direct_df.head(10).to_string(index=False))
print(f"\nSaved: {processed_l2_path}")

# ---- Integrity checks ----
print()

# 1. Max recurrence
max_n_cohorts_l2 = l2_summary_l2_direct["n_cohorts"].max()
assert max_n_cohorts_l2 >= 1, f"Expected at least 1 cohort recurrence, got {max_n_cohorts_l2}"
print(f"OK: max n_cohorts (direct-L2) = {max_n_cohorts_l2}/{len(COHORTS)}")

# 2. L2_importance sum within each cohort equals the sum of that cohort's top-K L2
#    normalized importances.
for cohort in COHORTS:
    cohort_l2_sum = matrix_df_l2_direct[
        matrix_df_l2_direct["cohort"] == cohort
    ]["L1_importance"].sum()
    # Recompute from scratch.
    l2_shap = pd.read_parquet(SHAP_DIR / cohort / f"{cohort}_l2_pathway_importance.parquet")
    l2_imp_by_name = l2_shap.groupby("pathway")["importance"].apply(lambda x: x.abs().mean())
    l2_imp = pd.Series(
        l2_imp_by_name.reindex(l2_names.values).fillna(0.0).to_numpy(),
        index=l2_names.index,
    )
    l2_imp_norm = _normalize_l2_imp(l2_imp)
    topK_sum = l2_imp_norm.nlargest(TOP_K_L2).sum()
    assert np.isclose(cohort_l2_sum, topK_sum), (
        f"{cohort}: L2_importance sum {cohort_l2_sum:.6f} != top-{TOP_K_L2} L2 sum {topK_sum:.6f}"
    )
print(f"OK: L2_importance sums match top-{TOP_K_L2} L2 normalized importance sums for all {len(COHORTS)} cohorts")

# 3. n_cohorts values are consistent.
check_nc_l2 = matrix_df_l2_direct.groupby("l1_id")["cohort"].nunique().rename("n_cohorts_check")
check_nc_l2 = check_nc_l2.reindex(l2_summary_l2_direct["l1_id"]).to_numpy()
mismatch_l2 = (l2_summary_l2_direct["n_cohorts"].to_numpy() != check_nc_l2)
assert not mismatch_l2.any(), (
    f"n_cohorts mismatch for: {l2_summary_l2_direct.loc[mismatch_l2, 'L1'].tolist()}"
)
print("OK: n_cohorts is consistent between matrix_df_l2_direct and l2_summary_l2_direct")